# EA2 - Actividad 2.6: Visualizacion de Datos

## Objetivos
- Convertir resultados de Spark a Pandas para visualizar
- Crear graficos de barras, histogramas y lineas con matplotlib
- Usar seaborn para heatmaps y boxplots
- Combinar multiples graficos en dashboards con subplots

## Conceptos Clave

### De Spark a Pandas: `toPandas()`

Spark procesa datos de forma distribuida, pero las librerias de visualizacion (matplotlib, seaborn) trabajan con datos locales en memoria. El metodo `toPandas()` convierte un DataFrame de Spark a un DataFrame de Pandas.

**Importante:** `toPandas()` trae **todos los datos al driver** (memoria local). Usarlo solo con:
- Resultados agregados (pocos registros)
- Subconjuntos pequenos de datos
- **NUNCA** con millones de registros sin agregar primero

### Tipos de Graficos

| Tipo | Uso | Libreria |
|------|-----|----------|
| **Barras** | Comparar categorias | matplotlib |
| **Histograma** | Distribucion de una variable | matplotlib |
| **Lineas** | Tendencias temporales | matplotlib |
| **Heatmap** | Correlaciones entre variables | seaborn |
| **Boxplot** | Distribucion y outliers | seaborn |
| **Subplots** | Dashboard con multiples graficos | matplotlib |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder.appName("visualizacion").master("local[*]").getOrCreate()

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Spark version: {spark.version}")

## 1. Cargar Datos

Usaremos los datasets de ventas (sales.csv) y tiendas (stores.csv), combinandolos con un join.

In [ ]:
# Leer datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

# Combinar con join
df = df_sales.join(df_stores, "Store")

print(f"Registros de ventas: {df_sales.count()}")
print(f"Tiendas: {df_stores.count()}")
print(f"Registros combinados: {df.count()}")
df.printSchema()

In [ ]:
# Vista previa de los datos
df.show(5)

## 2. Conversion a Pandas con `toPandas()`

> **Advertencia:** `toPandas()` trae todos los datos al driver local. Siempre agrega los datos **antes** de convertir para evitar problemas de memoria. Nunca uses `toPandas()` directamente sobre un DataFrame con millones de filas.

In [ ]:
# CORRECTO: Agregar primero, luego convertir (pocos registros)
df_tipo = df.groupBy("Type").agg(
    F.sum("Weekly_Sales").alias("total")
).toPandas()

print(f"Registros en Pandas: {len(df_tipo)}")
print(df_tipo)

## 3. Grafico de Barras con Matplotlib

Los graficos de barras son ideales para comparar valores entre categorias.

In [ ]:
# Grafico de barras: Ventas totales por tipo de tienda
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_tipo["Type"], df_tipo["total"], color=["#2196F3", "#4CAF50", "#FF9800"])
ax.set_title("Ventas Totales por Tipo de Tienda", fontsize=14)
ax.set_xlabel("Tipo")
ax.set_ylabel("Ventas Totales ($)")
plt.tight_layout()
plt.show()

## 4. Histograma de Distribucion

Los histogramas muestran como se distribuyen los valores de una variable numerica.

In [ ]:
# Preparar datos: muestra de ventas semanales
df_hist = df.select("Weekly_Sales").filter(F.col("Weekly_Sales") > 0) \
    .sample(fraction=0.1, seed=42).toPandas()

# Histograma de distribucion de ventas
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_hist["Weekly_Sales"], bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
ax.set_title("Distribucion de Ventas Semanales", fontsize=14)
ax.set_xlabel("Ventas Semanales ($)")
ax.set_ylabel("Frecuencia")
ax.axvline(df_hist["Weekly_Sales"].mean(), color="red", linestyle="--", label=f"Media: ${df_hist['Weekly_Sales'].mean():,.0f}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Grafico de Lineas (Serie Temporal)

Los graficos de lineas son perfectos para visualizar tendencias a lo largo del tiempo.

In [ ]:
# Preparar datos: ventas promedio por fecha
df_temporal = df.groupBy("Date").agg(
    F.avg("Weekly_Sales").alias("avg_sales")
).orderBy("Date").toPandas()

# Convertir fecha a datetime
df_temporal["Date"] = pd.to_datetime(df_temporal["Date"])

# Grafico de lineas
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_temporal["Date"], df_temporal["avg_sales"], color="#2196F3", linewidth=1.5)
ax.set_title("Tendencia de Ventas Promedio Semanales", fontsize=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas Promedio ($)")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Seaborn: Heatmap de Correlacion

El heatmap muestra la correlacion entre variables numericas. Valores cercanos a 1 o -1 indican correlacion fuerte.

In [ ]:
# Preparar datos para correlacion
df_corr = df.select(
    "Weekly_Sales", "Size", "Temperature", "Fuel_Price", "CPI", "Unemployment"
).toPandas()

# Heatmap de correlacion
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, cmap="coolwarm", ax=ax, 
            fmt=".2f", linewidths=0.5, center=0)
plt.title("Matriz de Correlacion", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Seaborn: Boxplot

Los boxplots muestran la distribucion de una variable, incluyendo la mediana, cuartiles y valores atipicos (outliers).

In [ ]:
# Preparar datos para boxplot
df_box = df.select("Type", "Weekly_Sales") \
    .filter(F.col("Weekly_Sales").between(0, 100000)) \
    .sample(fraction=0.1, seed=42).toPandas()

# Boxplot de ventas por tipo de tienda
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_box, x="Type", y="Weekly_Sales", palette="Set2", ax=ax)
ax.set_title("Distribucion de Ventas por Tipo de Tienda", fontsize=14)
ax.set_xlabel("Tipo de Tienda")
ax.set_ylabel("Ventas Semanales ($)")
plt.tight_layout()
plt.show()

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Top 10 departamentos por ventas totales
# =============================================================
# Grafico de barras horizontales con los top 10 departamentos

# 1. Agregar ventas por departamento
df_dept = df.groupBy("Dept").agg(
    F.sum("Weekly_Sales").alias("total")
).orderBy(F.desc("total")).limit(10).toPandas()

# Ordenar para que el mas alto quede arriba en el grafico horizontal
df_dept = df_dept.sort_values("total", ascending=True)

# 2. Crear grafico de barras horizontales
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_dept["Dept"].astype(str), df_dept["total"], color="#2196F3", edgecolor="white")

# 3. Agregar titulo y etiquetas
ax.set_title("Top 10 Departamentos por Ventas Totales", fontsize=14, fontweight="bold")
ax.set_xlabel("Ventas Totales ($)")
ax.set_ylabel("Departamento")

# Agregar valores en las barras
for i, (val, name) in enumerate(zip(df_dept["total"], df_dept["Dept"])):
    ax.text(val + val * 0.01, i, f"${val:,.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

> **Nota docente:** Puntos clave del ejercicio:
> - Se usa `ax.barh()` en lugar de `ax.bar()` para barras horizontales, lo cual
>   facilita la lectura de nombres largos en el eje Y.
> - `sort_values(ascending=True)` antes de graficar asegura que el departamento
>   con mas ventas quede arriba (matplotlib grafica de abajo hacia arriba).
> - `astype(str)` convierte los numeros de departamento a strings para que matplotlib
>   los trate como categorias, no como valores numericos.
> - Agregar valores en las barras con `ax.text()` es un detalle de presentacion
>   que mejora mucho la legibilidad (no es obligatorio pero suma puntos).
>
> Error comun: olvidar `limit(10)` en la query de Spark y traer todos los departamentos.

In [ ]:
# =============================================================
# EJERCICIO 2: Boxplot de ventas por tipo de tienda
# =============================================================
# Boxplot comparando Weekly_Sales por Type (A, B, C)

# 1. Preparar datos (filtrar outliers extremos y tomar muestra)
df_box2 = df.select("Type", "Weekly_Sales") \
    .filter(F.col("Weekly_Sales").between(0, 80000)) \
    .sample(fraction=0.1, seed=42).toPandas()

print(f"Registros para boxplot: {len(df_box2)}")

# 2. Crear boxplot con seaborn
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(
    data=df_box2, 
    x="Type", 
    y="Weekly_Sales", 
    palette="Set1",
    ax=ax,
    order=["A", "B", "C"]
)

# 3. Agregar titulo y etiquetas
ax.set_title("Distribucion de Ventas Semanales por Tipo", fontsize=14, fontweight="bold")
ax.set_xlabel("Tipo de Tienda")
ax.set_ylabel("Ventas Semanales ($)")

plt.tight_layout()
plt.show()

# Estadisticas descriptivas por tipo
print("\nEstadisticas por tipo:")
print(df_box2.groupby("Type")["Weekly_Sales"].describe().round(2))

> **Nota docente:** El boxplot permite comparar visualmente la distribucion entre grupos.
> Elementos del boxplot:
> - **Linea central:** Mediana (Q2)
> - **Caja:** Rango intercuartilico (Q1 a Q3, 50% central de los datos)
> - **Bigotes:** 1.5 * IQR desde Q1 y Q3
> - **Puntos:** Outliers (valores fuera de los bigotes)
>
> El filtro `between(0, 80000)` elimina outliers extremos que distorsionarian
> la visualizacion. Sin este filtro, la caja seria muy comprimida y los outliers
> dominarian el grafico.
>
> `order=["A", "B", "C"]` asegura un orden consistente en el eje X.
> Se pueden probar diferentes paletas: "Set1" (colores fuertes), "Set2" (suaves),
> "pastel" (muy suaves), "muted" (apagados).

In [ ]:
# =============================================================
# EJERCICIO 3: Tendencia mensual de ventas
# =============================================================
# Grafico de lineas con la tendencia mensual

# 1. Extraer anio y mes de la columna Date
df_mensual = df.withColumn("anio", F.year("Date")) \
    .withColumn("mes", F.month("Date"))

# 2. Agregar ventas por anio y mes
df_mensual = df_mensual.groupBy("anio", "mes").agg(
    F.sum("Weekly_Sales").alias("total")
).orderBy("anio", "mes").toPandas()

# 3. Crear columna de fecha para el eje X
df_mensual["fecha"] = pd.to_datetime(
    df_mensual["anio"].astype(str) + "-" + df_mensual["mes"].astype(str) + "-01"
)

print(f"Meses disponibles: {len(df_mensual)}")
print(df_mensual.head(10))

# 4. Crear grafico de lineas
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    df_mensual["fecha"], 
    df_mensual["total"], 
    marker="o", 
    color="#2196F3",
    linewidth=2,
    markersize=6
)

# 5. Titulo y etiquetas
ax.set_title("Tendencia Mensual de Ventas", fontsize=14, fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas Totales ($)")
ax.tick_params(axis='x', rotation=45)

# Agregar linea de promedio
promedio = df_mensual["total"].mean()
ax.axhline(y=promedio, color="red", linestyle="--", alpha=0.7, label=f"Promedio: ${promedio:,.0f}")
ax.legend()

plt.tight_layout()
plt.show()

> **Nota docente:** El grafico de tendencia mensual revela patrones estacionales
> en las ventas. Puntos clave:
> - `F.year()` y `F.month()` extraen componentes de fecha, lo cual permite
>   agregar por periodos.
> - Se construye una fecha "artificial" con el dia 1 de cada mes para tener
>   un eje X continuo y bien espaciado.
> - `marker="o"` muestra un punto en cada mes, lo cual facilita identificar
>   valores exactos.
> - La linea de promedio (`axhline`) da contexto sobre que meses estan por
>   encima o debajo del promedio general.
>
> Se deberia observar un pico en noviembre-diciembre (temporada navideña) y
> valores mas bajos en enero-febrero. Este patron es tipico del retail.
>
> Error comun: no convertir la fecha a datetime de Pandas, lo que causa que
> el eje X muestre strings en lugar de fechas bien formateadas.

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Dashboard con multiples graficos en una sola figura.

In [ ]:
# =============================================================
# DESAFIO: Dashboard con 4 graficos usando subplots(2,2)
# =============================================================

# Preparar los 4 DataFrames de Pandas
# (Reusamos los que ya tenemos arriba)

# Para el heatmap
df_corr_data = df.select(
    "Weekly_Sales", "Size", "Temperature", "Fuel_Price", "CPI", "Unemployment"
).toPandas()

# Para el boxplot
df_box_data = df.select("Type", "Weekly_Sales") \
    .filter(F.col("Weekly_Sales").between(0, 80000)) \
    .sample(fraction=0.1, seed=42).toPandas()

# Crear figura con subplots 2x2
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ==========================================
# Grafico (0,0) - Barras: Ventas por tipo
# ==========================================
axes[0, 0].bar(
    df_tipo["Type"], 
    df_tipo["total"], 
    color=["#2196F3", "#4CAF50", "#FF9800"]
)
axes[0, 0].set_title("Ventas por Tipo de Tienda", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Tipo")
axes[0, 0].set_ylabel("Ventas Totales ($)")

# ==========================================
# Grafico (0,1) - Linea: Tendencia temporal
# ==========================================
axes[0, 1].plot(
    df_mensual["fecha"], 
    df_mensual["total"], 
    color="#2196F3", 
    marker="o",
    linewidth=1.5,
    markersize=4
)
axes[0, 1].set_title("Tendencia Mensual de Ventas", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Fecha")
axes[0, 1].set_ylabel("Ventas Totales ($)")
axes[0, 1].tick_params(axis='x', rotation=45)

# ==========================================
# Grafico (1,0) - Heatmap: Correlacion
# ==========================================
sns.heatmap(
    df_corr_data.corr(), 
    annot=True, 
    cmap="coolwarm", 
    ax=axes[1, 0],
    fmt=".2f", 
    linewidths=0.5, 
    center=0,
    cbar_kws={"shrink": 0.8}
)
axes[1, 0].set_title("Matriz de Correlacion", fontsize=13, fontweight="bold")

# ==========================================
# Grafico (1,1) - Boxplot: Ventas por tipo
# ==========================================
sns.boxplot(
    data=df_box_data, 
    x="Type", 
    y="Weekly_Sales", 
    palette="Set2", 
    ax=axes[1, 1],
    order=["A", "B", "C"]
)
axes[1, 1].set_title("Distribucion de Ventas por Tipo", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Tipo de Tienda")
axes[1, 1].set_ylabel("Ventas Semanales ($)")

# Titulo general del dashboard
fig.suptitle("Dashboard de Ventas - Analisis Exploratorio", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

> **Nota docente:** El dashboard con subplots es una tecnica fundamental para
> presentar multiples perspectivas de los datos en una sola vista.
>
> Estructura clave:
> - `fig, axes = plt.subplots(2, 2, figsize=(16, 12))` crea una grilla 2x2.
> - Cada subplot se accede con `axes[fila, columna]` (indexado desde 0).
> - Cada grafico usa `ax=axes[i, j]` para indicar donde dibujar.
> - `fig.suptitle()` agrega un titulo general sobre todos los subplots.
> - `plt.tight_layout()` ajusta el espaciado automaticamente.
>
> Buenas practicas para dashboards:
> 1. Cada grafico debe tener titulo propio y ejes etiquetados.
> 2. Usar colores consistentes (el mismo azul para barras y lineas).
> 3. El `figsize` debe ser lo suficientemente grande para que todo sea legible.
> 4. Preparar todos los DataFrames **antes** de crear la figura.
>
> Error comun: confundir `plt.title()` (afecta al ultimo subplot activo) con
> `ax.set_title()` (afecta al subplot especifico). Siempre usar `ax.set_title()`
> cuando se trabaja con subplots.

---
## Resumen

En esta actividad aprendimos:

1. **toPandas():** Convertir DataFrames de Spark a Pandas (solo con datos agregados/pequenos)
2. **Graficos de barras:** `ax.bar()` y `ax.barh()` para comparar categorias
3. **Histogramas:** `ax.hist()` para ver distribuciones de variables
4. **Graficos de lineas:** `ax.plot()` para tendencias temporales
5. **Heatmap:** `sns.heatmap()` para visualizar correlaciones
6. **Boxplot:** `sns.boxplot()` para distribucion y outliers
7. **Subplots:** `plt.subplots(filas, columnas)` para dashboards con multiples graficos
8. **Buenas practicas:** Siempre agregar datos antes de `toPandas()` para evitar problemas de memoria

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")